In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
language_role_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/language_role.json")
    )

languageRole,roleId
Original,1
Spoken,2


In [0]:
language_role_schema = StructType([
    StructField("languageRole", StringType()),
    StructField("roleId", IntegerType())
])

language_role_df = (
    spark.read
        .schema(language_role_schema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/language_role.json")
)

languageRole,roleId
Original,1
Spoken,2


In [0]:
language_role_df = (
    language_role_df
        .withColumnsRenamed(
            {
                "languageRole": "language_role",
                "roleId": "role_id"
            }

        )
        .withColumns(
            {
                "env": lit(v_env),
                "file_date": lit(v_file_date)
            }
        )
)

language_role_df = add_ingestion_date(language_role_df)

language_role,role_id,env,file_date,ingestion_date
Original,1,dev,2024-12-16,2026-10-08T03:44:42.596Z
Spoken,2,dev,2024-12-16,2026-10-08T03:44:42.596Z


In [0]:
(
    language_role_df
        .write
            .mode("overwrite")
            .saveAsTable("movie_silver.language_roles")
)

In [0]:
dbutils.notebook.exit("Success")